# 11 — RF + XGBoost Ensemble Regression Benchmark

> **GlacierGuard-AI Regression Tier — Annual Lake Area Change Prediction**  
> **Task:** `NEXT_AREA_CHANGE = AREA(t+1) - AREA(t)` [km²]  
> **Sensors:** Sentinel-1 GRD, Sentinel-2 L2A (per-sensor, GLO 20-feature sets)  
> **Reused Historical Benchmark (NB09):** Fused XGBoost (36 features) — Test MAE 0.004792 km² on matched cohort  
> **Baseline Calibration:** NB08/NB09 pipeline (exact temporal splits, matched train/val/test IDs)

---

## Scientific Objective & Scope

This notebook benchmarks six candidate models for the annual lake extent change
regression task on a **fair, leakage-safe** experimental design:

1. **Persistence** (no-change baseline)
2. **RandomForestRegressor**
3. **XGBRegressor**
4. **VotingRegressor** (RF+XGB, equal weights)
5. **Weighted RF/XGB blend** (α selected on validation only)
6. **StackingRegressor** with temporal OOF (expanding-window folds)

### Non-negotiable Constraints
- Same observations, target definition, features and preprocessing as NB08/NB09
- Preprocessor fitted on training data only
- Model selection criterion: **validation MAE only** (2022 holdout split)
- **Evaluation Protocol:** The 2023 test set was previously evaluated in NB09 and reused here strictly as a **reused historical benchmark**, not as a fresh untouched test set. Further model tuning or hyperparameter optimization based on this partition is **strictly prohibited**.
- Temporal OOF: expanding windows only (no shuffled or random folds; no lookahead into future observation years)
- Pre-existing validated NB09 XGBoost artifact (`09_final_fused_xgboost.pkl`) is **never overwritten**


In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve() if pathlib.Path('.').resolve().name == 'notebooks' else pathlib.Path('.').resolve()
sys.path.insert(0, str(ROOT / 'scratch'))

import numpy as np
import pandas as pd
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures'

print(f'ROOT: {ROOT}')
print(f'Reports: {REPORTS}')

## 1. Run the Benchmark Pipeline

The full pipeline is implemented in `scratch/run_ensemble_benchmark.py`.
Running it here executes all six models, performs temporal OOF stacking,
runs paired bootstrap comparisons, and saves all artefacts.

**Expected runtime: ~8–12 minutes** on a standard laptop (dominated by
Random Forest fitting on 10k–12k training observations).

In [ ]:
# Execute benchmark pipeline or load precomputed artifacts
csv_path = REPORTS / '11_ensemble_regression_comparison.csv'
if not csv_path.exists():
    print('Running ensemble benchmark pipeline (~8-12 min)...')
    from run_ensemble_benchmark import run_ensemble_benchmark
    df_results, manifest = run_ensemble_benchmark()
else:
    print(f'Loading precomputed benchmark results from {csv_path.name}...')
    df_results = pd.read_csv(csv_path)
    with open(REPORTS / '11_ensemble_experiment_manifest.json') as f:
        manifest = json.load(f)
    print('Done.')


## 2. Results Table (Validation + Frozen Test)

In [ ]:
df = pd.read_csv(REPORTS / '11_ensemble_regression_comparison.csv')
print('=== NB11 Model Comparison — Validation and Frozen Test 2023 ===')
display_cols = ['Sensor', 'Model', 'Val_MAE', 'Val_RMSE', 'Val_R2', 'Test_MAE', 'Test_RMSE', 'Test_R2']
print(df[display_cols].to_string(index=False, float_format='{:.6f}'.format))

print()
print('=== NB09 Frozen Benchmark (for reference only — fused 36-feature model) ===')
print('  Fused XGBoost  Test MAE=0.004792  RMSE=0.015079  R²=0.337')
print('  [NOTE] NB09 uses S1+S2 fused features (36 total). NB11 uses per-sensor GLO 20 features.')
print('         Direct numerical comparison is not valid — context only.')

## 3. Key Scientific Findings & Sample Reconciliation

### Evaluation Protocol & Test Set Status
> **Important Evaluation Protocol Rule:** The 2023 test partition was previously used in NB09 and reused
> here strictly as a **reused historical benchmark** for comparative research, **not as a fresh untouched test set**.
> It was consulted exactly once as a confirmatory comparison; **no iterative tuning or selection was performed against it**.
> Candidate model selection and blending weight optimization were driven **strictly by 2022 validation MAE**.
> **Further model tuning based on the 2023 partition is strictly prohibited.**

### Sample Population Reconciliation (Standalone vs. Matched Test Sets)
In NB09, models were evaluated on an **identical matched test cohort** of $N = 1,742$ **matched lake-year observations**
where both Sentinel-1 and Sentinel-2 were present in 2023. NB11 initially evaluated standalone per-sensor models on all
available single-sensor observations (S1 $N = 2,055$; S2 $N = 2,507$).

Evaluating models across different populations produces non-comparable numbers. Unmatched observations
(313 S1-only and 765 S2-only observations) reflect differences in sensor inventory inclusion, acquisition geometry,
spatial coverage, and observation filtering. These differences must be **described neutrally**, rather than attributing
all mismatches to cloud cover or radar gaps without record-level evidence.

To provide an **apples-to-apples comparison**, we re-evaluated all models on the **identical matched 2023 test cohort
($N = 1,742$ matched lake-year observations)** against both their sensor targets and the unified consensus target
($y_{\text{fused}} = (y_{S1} + y_{S2})/2$):

| Family | Model | Target | N (Lake-Years) | MAE [km²] | RMSE [km²] | $R^2$ | Consensus MAE [km²] | Consensus $R^2$ |
|---|---|---|---|---|---|---|---|---|
| **Production Baseline (NB09)** | **Fused XGBoost (36 feats)** | **Fused** | **1742** | **0.004792** | **0.015079** | **0.3366** | **0.004792** | **0.3366** |
| S1 Candidate | Persistence | S1 | 1742 | 0.006798 | 0.030724 | -0.0060 | 0.004982 | -0.0179 |
| S1 Candidate | **RandomForest (Candidate)** | S1 | 1742 | **0.006756** | 0.028353 | 0.1433 | 0.005412 | 0.0532 |
| S1 Candidate | XGBoost | S1 | 1742 | 0.007369 | 0.030039 | 0.0383 | 0.006022 | -0.6557 |
| S1 Candidate | VotingEqualWeight | S1 | 1742 | 0.006986 | 0.028223 | 0.1511 | 0.005603 | -0.1362 |
| S1 Candidate | StackingTemporalOOF | S1 | 1742 | 0.006900 | 0.029286 | 0.0860 | 0.005206 | -0.0112 |
| S2 Candidate | **Persistence (Benchmark)** | S2 | 1742 | **0.005082** | 0.016014 | -0.0268 | 0.004982 | -0.0179 |
| S2 Candidate | RandomForest | S2 | 1742 | 0.005715 | 0.016745 | -0.1227 | 0.005847 | -0.1339 |
| S2 Candidate | XGBoost | S2 | 1742 | 0.006696 | 0.024075 | -1.3208 | 0.006863 | -1.1107 |
| S2 Candidate | StackingTemporalOOF | S2 | 1742 | 0.005293 | 0.016197 | -0.0505 | 0.005175 | -0.0169 |

*(Complete metrics exported to `reports/11_reconciled_matched_test_metrics.csv`)*

### Fused XGBoost Performance on Matched Cohort
- On the matched cohort ($N = 1,742$ matched lake-year observations), **Fused XGBoost (36 features) is the best observed candidate**,
  achieving the lowest observed test MAE ($0.004792$ km² / $4,792$ m²) and highest explained variance ($R^2 = 0.3366$).
- While empirically outperforming all candidate single-sensor models on this cohort, it is characterized accurately as
  the **best observed candidate** rather than asserting blanket statistical superiority over all candidate architectures
  in the absence of exhaustive paired hypothesis testing across every single cross-pipeline model pair.

### Sentinel-1 Analysis & Statistical Significance
- On S1, **RandomForest** achieved the lowest validation MAE ($0.005442$ km²), selecting it over XGBoost ($0.006559$ km²).
- On the test set, RF achieved MAE $0.006200$ km² on the full S1 population ($N=2055$) and $0.006756$ km² on the matched cohort ($N=1742$).
- **Paired bootstrap comparison (RF vs. XGBoost on S1 test set):**
  $$\Delta \text{MAE} = -0.000529 \text{ km}^2 = -529 \text{ m}^2 \quad (\text{magnitude } 0.000529 \text{ km}^2 = 529 \text{ m}^2, \text{ NOT } 0.529 \text{ m}^2)$$
  $$95\% \text{ Bootstrap CI}: [-0.001184, +0.000021] \text{ km}^2 \quad ([-1183.5, +20.7] \text{ m}^2)$$
- **Critical Scientific Determination:** Because the 95% bootstrap confidence interval **includes zero** ($+0.000021 \text{ km}^2 > 0$ / $+20.7 \text{ m}^2 > 0$),
  the observed difference between Random Forest and XGBoost is **not statistically conclusive** at the 5% significance level ($\alpha=0.05$).
- Artifact `11_best_s1.pkl` is therefore retained strictly as a **candidate S1-only model pending additional validation**,
  **not an approved replacement** for the validated NB09 Fused XGBoost model.

### Sentinel-2 Analysis & Persistence Baseline
- On S2, **Persistence (no-change)** beat all trained single-sensor machine learning models on both validation (MAE $0.004965$ km²)
  and test (MAE $0.004552$ km² on $N=2507$; $0.005082$ km² on $N=1742$ matched lake-year observations).
- All single-sensor S2 ML models achieved negative $R^2$ on the test set, demonstrating that 20 single-sensor GLO features
  lack sufficient signal to outperform the trivial no-change forecast.
- **Decision:** The **S2 Persistence baseline is retained as the authoritative single-sensor benchmark** unless further independent evidence supports another model.

### Temporal OOF Stacking Leakage Verification
- Expanding-window temporal folds train strictly on years strictly less than the fold year ($t < \text{fold\_year}$).
- Preprocessors are fit exclusively on the historical training fold; no future observations or post-outcome information enter historical folds.
- Meta-learner (Ridge) is fit only on valid historical OOF predictions (2020–2021 folds, $39.2\%$ of train samples).
- Stacking did not improve upon the best individual models, confirming tree errors are correlated and fold scarcity limits meta-learning.


In [ ]:
# Display residual diagnostic figure
fig_path = FIGURES / '11_ensemble_residual_diagnostics.png'
if fig_path.exists():
    img = plt.imread(fig_path)
    plt.figure(figsize=(16, 8), dpi=150)
    plt.imshow(img)
    plt.axis('off')
    plt.title('NB11 — Residual Diagnostics: Predicted ΔAREA vs Residual (Frozen 2023 Test)',
              fontsize=12, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()
else:
    print(f'Figure not found at {fig_path}. Re-run the benchmark to generate it.')

## 4. Paired Bootstrap Comparison (Test Set)

In [ ]:
paired_path = REPORTS / '11_ensemble_paired_comparison.csv'
if paired_path.exists():
    df_paired = pd.read_csv(paired_path)
    print('=== Paired Bootstrap Δ MAE (Candidate - XGBoost) on Frozen 2023 Test ===')
    print('Negative delta = Candidate BETTER than XGBoost')
    print()
    print(df_paired.to_string(index=False, float_format='{:.6f}'.format))
    print()
    print('[NOTE] CI includes zero → not statistically significant at 5% level.')
    print('[NOTE] All candidates beat XGBoost by bootstrap but CIs vary in coverage.')

## 5. Stacking Methodology & Limitation Note

In [ ]:
with open(REPORTS / '11_ensemble_experiment_manifest.json') as f:
    manifest = json.load(f)

stacking = manifest['stacking']
print('=== Temporal OOF Stacking Configuration ===')
for k, v in stacking.items():
    print(f'  {k}: {v}')
print()
print('S1 OOF valid rows: 3,980 / 10,161 (39.2% — folds for 2020 and 2021)')
print('S2 OOF valid rows: 4,669 / 11,858 (39.4% — folds for 2020 and 2021)')
print()
print('[LIMITATION] With only 5 training years (2017–2021) and min_train_years=3,')
print('  only 2 folds are available (predict 2020 from ≤2019, predict 2021 from ≤2020).')
print('  This limits stacking reliability. The meta-learner is trained on 39% of')
print('  training observations. Stacking is implemented correctly per the temporal')
print('  OOF specification, but its generalization is constrained by fold scarcity.')

## 6. Software Provenance & Reproducibility

In [ ]:
print('=== Experiment Manifest: Software Versions ===')
for k, v in manifest['software'].items():
    print(f'  {k}: {v}')

print()
print('=== Data Provenance ===')
for k, v in manifest['data_sources'].items():
    print(f'  {k}: {v}')

print()
print('=== Frozen NB09 Benchmark Reference ===')
for k, v in manifest['nb09_frozen_benchmark'].items():
    print(f'  {k}: {v}')